## Data Loader
[인공지능 기술·산업 생태계 육성방안
연구](https://spri.kr/posts/view/23669)



In [17]:
from settings import get_settings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

settings = get_settings()

def create_model()-> ChatOpenAI:
    return ChatOpenAI(
        model=settings.default_model,
        base_url=settings.base_url,
        api_key=settings.api_key,
    )


def create_embeddings() -> OpenAIEmbeddings:
    return OpenAIEmbeddings(
        model=settings.embedding_model,
        base_url=settings.base_url,
        api_key=settings.api_key,
        check_embedding_ctx_length=False
    )

llm = create_model()
embeddings = create_embeddings()


설정

In [18]:
import pdfplumber
from pathlib import Path

FILE_PATH = Path.cwd().parent/"data"/"RE-185. 인공지능 기술·산업 생태계 육성방안 연구.pdf"

text = ""

with pdfplumber.open(FILE_PATH) as pdf:
    for page in pdf.pages:
        text += page.extract_text()

In [19]:
text[:50]

'연구보고서 RE-185\n인공지능 기술·산업 생태계 육성방안\n연구\nA Study on the'

In [20]:
from utils import chunk_text

chunks = chunk_text(text, 500, 40)
print(len(chunks))
print(chunks[0])


290
연구보고서 RE-185
인공지능 기술·산업 생태계 육성방안
연구
A Study on the Promotion Policy for the Korean Technological and
Industirial Ecosystem in Artificial Intelligence
봉강호 / 안성원
2025. 4.이 보고서는 2024년도 과학기술정보통신부 정보통신진흥기금을 지원
받아 수행한 연구결과로 보고서 내용은 연구자의 견해이며, 과학기술정보
통신부의 공식입장과 다를 수 있습니다.목 차
제1장 서론 ·························································································································· 1
제1절 연구 배경 및 필요성 ··························································································


In [70]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
)

chunks = text_splitter.split_text(text)
print(len(chunks))
print(chunks[0])

163
연구보고서 RE-185
인공지능 기술·산업 생태계 육성방안
연구
A Study on the Promotion Policy for the Korean Technological and
Industirial Ecosystem in Artificial Intelligence
봉강호 / 안성원
2025. 4.이 보고서는 2024년도 과학기술정보통신부 정보통신진흥기금을 지원
받아 수행한 연구결과로 보고서 내용은 연구자의 견해이며, 과학기술정보
통신부의 공식입장과 다를 수 있습니다.목 차
제1장 서론 ·························································································································· 1
제1절 연구 배경 및 필요성 ·························································································· 1
제2절 연구 목적 및 내용 ······························································································ 5
제2장 주요국 정책 추진 현황 분석 ·············································································· 6
제1절 국가별 현황 ·········································································································· 6
1. 미국········································································································································· 6


### PDF Text Preprocessing 1
PDF에서 추출한 텍스트에는 목차의 점선 리더(dot leader)가 포함되어 있다.
```text
제1장 서론 ··········································· 1
```
이러한 문자열은 문서의 의미를 가지지 않으며, 청킹 및 임베딩 시 불필요한 토큰을 증가시키고 검색 품질을 저하시킬 수 있다.
따라서 정규식을 이용하여 연속된 점선(`···`, `...`)을 제거하는 전처리를 수행한다.


In [71]:
import re

def clean_pdf_text(text: str) -> str:
    text = re.sub(r"[·.]{3,}", " ", text)
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

In [74]:
clean_text = clean_pdf_text(text)
chunks = text_splitter.split_text(clean_text)
print(len(chunks))
print(chunks[0])

155
연구보고서 RE-185
인공지능 기술·산업 생태계 육성방안
연구
A Study on the Promotion Policy for the Korean Technological and
Industirial Ecosystem in Artificial Intelligence
봉강호 / 안성원
2025. 4.이 보고서는 2024년도 과학기술정보통신부 정보통신진흥기금을 지원
받아 수행한 연구결과로 보고서 내용은 연구자의 견해이며, 과학기술정보
통신부의 공식입장과 다를 수 있습니다.목 차
제1장 서론 1
제1절 연구 배경 및 필요성 1
제2절 연구 목적 및 내용 5
제2장 주요국 정책 추진 현황 분석 6
제1절 국가별 현황 6
1. 미국 6
2. 중국 14
3. EU 23
4. 싱가포르 30
제2절 소결 및 시사점 38
제3장 글로벌 AI 연구 현황 분석 42
제1절 데이터 수집 방법론 개발 필요성 42
제2절 데이터 수집 방법론 43
1. AI 기술 키워드 도출 및 검색식 작성 43
2. AI 분야 학술논문 수집 45
제3절 글로벌 AI 분야 논문 현황 및 시사점 50
제4장 국내·외 AI 기업 분석 및 사례 연구 53
제1절 개요 53
제2절 AI 기업의 경제적 성과 영향요인에 관한 실증분석 56
1. 선행연구 검토 56
2. 연구자료 및 방법 58
3. 분석 결과 및 시사점 63
제3절 국내·외 AI 기업 우수사례 분석 67
1. 연구 방법 및 대상 67
2. 조사·분석 결과 70
3. 소결 및 시사점 79제5장 글로벌 AI 시장 동향 및 국내 AI 수요 현황 분석 82
제1절 개요 및 AI 시장 동향 82
제2절 국내 기업의 AI 수요 현황 및 인식 조사 86
1. 조사 개요 86
2. 조사 결과 90
제3절 소결 및 시사점 95
제6장 결론 97
제1절 연구결과 종합 97
제2절 영역별 정책제언 100
참고문헌 105
부록(AI 기술 관련 논문 검색 키워드 조합) 111
- iv -표 목 차
<표 1-1> 국가별 AI R&D 전략 추진 동향 3


### PDF Text Preprocessing 2

PDF 텍스트에서 점선 리더(dot leader)를 제거한 뒤에도 다음과 같은 목차 항목이 남아있다.
```text
제1절 데이터 수집 방법론 개발 필요성 42
```
- 위의 마지막 숫자 42는 페이지 번호이며, 전체 문장은 본문이 아니라 목차의 항목에 해당한다.
- 페이지 번호만 제거하면 다음과 같이 목차 제목이 여전히 청크에 포함된다.
```text
제1절 데이터 수집 방법론 개발 필요성
```
목차 항목은 실제 본문 내용이 아니며, 벡터 검색 시 동일한 제목이 본문보다 우선 검색되거나 불필요한 청크가 생성될 수 가능성이 있다.
1. 페이지 번호만 제거하지 않고, 점선 리더와 페이지 번호로 구성된 목차 항목은 해당 줄 전체를 제거?
2. 목차 항목을 정규식으로 개별 제거하지 않고, 두 번째 목차가 포함된 페이지 전체를 문서 처리 대상에서 제외?

```
3. text = re.sub(
    r"^.*[·.]{3,}\s*\d+\s*$",
    "",
    text,
    flags=re.MULTILINE,
)
```

In [75]:
print(len(chunks))

155


In [76]:
vectors = embeddings.embed_documents(chunks)

print(vectors[0][:3])
print(len(vectors))
print(len(vectors[0]))

[-0.04780101031064987, 0.01958061195909977, -0.026092398911714554]
155
1024


Neo4j Driver 연결한다.

In [67]:
from neo4j import GraphDatabase
driver = GraphDatabase.driver(settings.neo4j_url, auth=(settings.neo4j_user, settings.neo4j_password))

try:
    driver.verify_connectivity()
    print("Neo4j 연결 성공")
except Exception as e:
    print(f"연결 실패: {e}")

Neo4j 연결 성공


In [69]:
# query_path = (
#     Path.cwd()
#     / "cypher"
#     / "000-drop-node.cypher"
# )
driver.execute_query(
    "MATCH (n) DETACH DELETE n",
    database_="neo4j",
)

driver.execute_query(
    "DROP INDEX pdf IF EXISTS",
    database_="neo4j",
)
#
# query = query_path.read_text(encoding="utf-8")
# driver.execute_query(query, database_="neo4j")

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x13fd8deb0>, keys=[])

Neo4J Index를 생성한다.
```cypher
CREATE VECTOR INDEX pdf
IF NOT EXISTS FOR (c:Chunk) ON c.embedding
```

In [30]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
CYPHER_DIR = PROJECT_ROOT / "cypher"

query_path = CYPHER_DIR / "001-create-vector-index.cypher"


In [ ]:
query_path = (
    Path.cwd()
    / "cypher"
    / "001-create-vector-index.cypher"
)

query = query_path.read_text(encoding="utf-8")
driver.execute_query(query, database_="neo4j")


In [34]:
query_path = (
    Path.cwd()
    / "cypher"
    / "002-show-index.cypher"
)

query = query_path.read_text(encoding="utf-8")
records, summary, keys = driver.execute_query(query, database_="neo4j")

In [36]:
records

[<Record name='pdf' state='ONLINE' labelsOrTypes=['Chunk'] properties=['embedding'] type='VECTOR'>]

In [37]:
summary

In [38]:
# 레코드 확인
for record in records:
    print(record.data())

{'name': 'pdf', 'state': 'ONLINE', 'labelsOrTypes': ['Chunk'], 'properties': ['embedding'], 'type': 'VECTOR'}


Neo4j 저장

In [39]:
query_path = (
    Path.cwd()
    / "cypher"
    / "003-create-chunk-node.cypher"
)

query = query_path.read_text(encoding="utf-8")


In [41]:
embedding_vectors = embeddings.embed_documents(chunks)

In [42]:
driver.execute_query(query, chunks=chunks, embeddings=embedding_vectors)

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x12a457e60>, keys=[])

In [44]:
query_path = (
    Path.cwd()
    / "cypher"
    / "004-match-all.cypher"
)

query = query_path.read_text(encoding="utf-8")

records, _, _ = driver.execute_query(query)

print(records[0]["c.text"][0:30])
print(records[0]["c.embedding"][0:3])

연구보고서 RE-185
인공지능 기술·산업 생태계 육성
[-0.05807943269610405, 0.011216574348509312, -0.007120043970644474]


In [46]:
question = "인공지능 산업 생태계 육성 방안의 목적은 무엇인가?"
question_embedding = embeddings.embed_query(question)

print(len(question_embedding))

1024


In [55]:
query_path = (
    Path.cwd()
    / "cypher"
    / "005-query-vector-index.cypher"
)

query = query_path.read_text(encoding="utf-8")
similar_records, _, _ = driver.execute_query(query, question_embedding=question_embedding, k=4)

for i, record in enumerate(similar_records, start=1):
    print(f"{i} {'-' * 20}")
    print(record["text"])
    print(record["score"], record["node"]["index"])
    print("-"*20)

1 --------------------
연구보고서 RE-185
인공지능 기술·산업 생태계 육성방안
연구
A Study on the Promotion Policy for the Korean Technological and
Industirial Ecosystem in Artificial Intelligence
봉강호 / 안성원
2025. 4.이 보고서는 2024년도 과학기술정보통신부 정보통신진흥기금을 지원
받아 수행한 연구결과로 보고서 내용은 연구자의 견해이며, 과학기술정보
통신부의 공식입장과 다를 수 있습니다.목 차
제1장 서론 ·························································································································· 1
제1절 연구 배경 및 필요성 ··························································································
--------------------
2 --------------------
지능정보
기술과 산업 간 융합을 통해 새로운 경제발전 생태계를 창조하는 것을 본격적으로 국가
정책에서 강조하기 시작했다는 점에서 의미를 찾을 수 있다고 하겠다.
<표 2-10> 중국의 초기 인공지능 정책 주요 내용
구분 내용
§ ‘정보화와 산업화의 심층적인 통합 촉진’ 임무 내에서 지능형 제조를 위한
개발 전략 연구, 지능형 제조 장비·제품 개발 가속화, 제조 공정의 지능화
중국제조 2025 촉진 등 제시
(2015년 5월) § 정보통신설비를 기반으로 한 차세대 정보기술산업, 디지털 제어장치와
로봇, 항공장비, 해양공정장비와 하이테크 선박, 농기계 장비 등 AI 기반
으로 스마트화할 수 있는 제조업 분야를 중점 추진분야로 제시
인터넷+ § 인터넷 서비스와 경제·사회 각 분야의 융합 발전을 통한 신성장동력 창출을
행동계획 위한 11대 중점 과제를 제시


In [56]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
당신은 AI 기술 및 산업 정책 연구 전문가입니다.

다음 규칙을 반드시 지켜 답변하세요.
- 제공된 문서 내용만 근거로 답변합니다.
- 문서에 없는 내용은 추측하거나 임의로 생성하지 않습니다.
- 답을 찾을 수 없는 경우 "제공된 문서에서 해당 내용을 찾을 수 없습니다."라고 답변합니다.
- 답변은 명확하고 간결하게 작성합니다.
            """.strip(),
        ),
        (
            "human",
            """
다음은 검색된 문서입니다.

{context}

---

위 문서에 포함된 정보만 사용하여 아래 질문에 답변하세요.

질문:
{question}
            """.strip(),
        ),
    ]
)

In [ ]:
similar_records, _, _ = driver.execute_query(query, question_embedding=question_embedding, k=4)

In [58]:
context = "\n\n".join(doc["text"] for doc in similar_records)
#question = "인공지능 산업 생태계 육성의 목적은 무엇인가?"
question = "보고서에서 중요하게 다루는 AI 핵심 기술은 무엇인가?"
messages = prompt.invoke(
    {
        "context": context,
        "question": question,
    }
)

response = llm.invoke(messages)

print(response.content)

제공된 문서에 따르면, 보고서는 다음과 같은 AI 기술들을 다루고 있습니다.

*   **기초 AI R&D:** 머신러닝, 컴퓨터비전
*   **첨단 AI:** 생성형 AI, 자율주행, 로보틱스


In [62]:
from pathlib import Path
from langchain_core.prompts import ChatPromptTemplate


query_path = (
    Path.cwd()
    / "cypher"
    / "005-query-vector-index.cypher"
)

vector_search_query = query_path.read_text(encoding="utf-8")


# 2. RAG 프롬프트 템플릿
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
당신은 AI 기술 및 산업 정책 연구 전문가입니다.

다음 규칙을 반드시 지켜 답변하세요.
- 제공된 문서 내용만 근거로 답변하세요.
- 문서에 없는 내용은 추측하지 마세요.
- 답을 찾을 수 없다면
  "제공된 문서에서 해당 내용을 찾을 수 없습니다."라고 답변하세요.
- 답변은 한국어로 명확하게 작성하세요.
            """.strip(),
        ),
        (
            "human",
            """
다음은 질문과 관련하여 검색된 문서입니다.

{context}

---

위 문서에 포함된 정보만 사용하여 아래 질문에 답변하세요.

질문:
{question}
            """.strip(),
        ),
    ]
)


# 3. 질문부터 LLM 답변까지 한 번에 처리
def ask_rag(question: str, k: int = 4) -> str:
    question_embedding = embeddings.embed_query(question)


    similar_records, _, _ = driver.execute_query(
        vector_search_query,
        question_embedding=question_embedding,
        k=k,
        database_="neo4j",
    )

    if not similar_records:
        return "관련 문서를 찾을 수 없습니다."

    context = "\n\n".join(
        f"[문서 {i + 1} | 유사도: {record['score']:.4f}]\n"
        f"{record['text']}"
        for i, record in enumerate(similar_records)
    )

    messages = prompt.invoke(
        {
            "context": context,
            "question": question,
        }
    )

    response = llm.invoke(messages)

    return response.content

In [63]:
question = "AI 전문 인력 양성을 위해 어떤 정책을 제안하고 있는가?"

print(ask_rag(question, k=5))

제시된 문서들은 AI 전문 인력 양성 및 확보를 위해 단기적 노력과 장기적 전략을 결합한 다각적인 정책들을 제안하고 있습니다.

**주요 제안 및 추진 정책은 다음과 같습니다:**

### Ⅰ. 인재 확보를 위한 투 트랙(Two-track) 전략
*   **국내 고급 인력 지원:** 자국 내 고급 인력의 연구 활동 및 창업을 적극적으로 지원하는 정책(예: 중국의 ‘만인계획’)을 펼치고 있습니다. 또한, 도메인 전문가들의 경력 전환을 위한 교육 및 지원을 확대할 필요가 있습니다.
*   **해외 인재 유치:** 해외 고급 인재를 적극적으로 유치하는 정책을 추진하고 있으며, 이는 단기적 확보 노력과 중장기적 인재 흡수 환경 조성이라는 두 가지 측면에서 접근되어야 합니다.

### Ⅱ. 교육 및 시스템 혁신을 통한 미래 대비
*   **AI 전문 교육 강화:** ① AI 발전에 최적화된 대학 시스템을 마련하고, ② 학과 구조조정 및 산·학·연/국제협력을 강화하여 AI 인재 교육 체계를 완비해야 합니다.
*   **전 계층의 디지털 역량 강화:** 단순히 AI 전문가를 양성하는 것을 넘어, 전 계층의 전반적인 디지털 문해력을 향상시키는 교육을 병행해야 합니다.
*   **생애주기별 지원:** 잠재력 있는 인재를 발굴하고, 다양한 교육과 실무 경험의 축적을 통해 성장할 수 있도록 생애주기별 지원 방안을 모색해야 합니다.

### Ⅲ. 정책 및 연구 환경의 시스템적 접근
*   **통합적이고 지속가능한 인재 정책:** 현재 당면한 인력 수급 정책과 미래의 인재를 양성하기 위한 전략 사이에 긴밀하게 연계성을 확보할 수 있는 통합적 정책 구상이 필요합니다. 우수한 인재는 단기간에 육성될 수 없으므로 장기적인 전략과 전폭적 지원이 요구됩니다.
*   **정책적 중요도 상향 조정:** 우수한 인재가 곧 국가의 경쟁력이라는 관점에서, 현재와 미래를 모두 고려하는 '시스템적' 접근을 통해 인재 관련 정책의 중요도를 대폭 상향 조정해야 합니다.
*   **과감한 투자:** 기초 연구